# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [2]:
%help


# Available Magic Commands

## Sessions Magic

----
    %help                             Return a list of descriptions and input types for all magic commands. 
    %profile            String        Specify a profile in your aws configuration to use as the credentials provider.
    %region             String        Specify the AWS region in which to initialize a session. 
                                      Default from ~/.aws/config on Linux or macOS, 
                                      or C:\Users\ USERNAME \.aws\config" on Windows.
    %idle_timeout       Int           The number of minutes of inactivity after which a session will timeout. 
                                      Default: 2880 minutes (48 hours).
    %timeout            Int           The number of minutes after which a session will timeout. 
                                      Default: 2880 minutes (48 hours).
    %session_id_prefix  String        Define a String that will precede all session IDs in the format 
                                      [session_id_prefix]-[session_id]. If a session ID is not provided,
                                      a random UUID will be generated.
    %status                           Returns the status of the current Glue session including its duration, 
                                      configuration and executing user / role.
    %session_id                       Returns the session ID for the running session.
    %list_sessions                    Lists all currently running sessions by ID.
    %stop_session                     Stops the current session.
    %glue_version       String        The version of Glue to be used by this session. 
                                      Currently, the only valid options are 2.0, 3.0 and 4.0. 
                                      Default: 2.0.
    %reconnect          String        Specify a live session ID to switch/reconnect to the sessions.
----

## Selecting Session Types

----
    %streaming          String        Sets the session type to Glue Streaming.
    %etl                String        Sets the session type to Glue ETL.
    %session_type       String        Specify a session_type to be used. Supported values: streaming and etl.
----

## Glue Config Magic 
*(common across all session types)*

----

    %%configure         Dictionary    A json-formatted dictionary consisting of all configuration parameters for 
                                      a session. Each parameter can be specified here or through individual magics.
    %iam_role           String        Specify an IAM role ARN to execute your session with.
                                      Default from ~/.aws/config on Linux or macOS, 
                                      or C:\Users\%USERNAME%\.aws\config` on Windows.
    %number_of_workers  int           The number of workers of a defined worker_type that are allocated 
                                      when a session runs.
                                      Default: 5.
    %additional_python_modules  List  Comma separated list of additional Python modules to include in your cluster 
                                      (can be from Pypi or S3).
    %%tags        Dictionary          Specify a json-formatted dictionary consisting of tags to use in the session.
    
    %%assume_role Dictionary, String  Specify a json-formatted dictionary or an IAM role ARN string to create a session 
                                      for cross account access.
                                      E.g. {valid arn}
                                      %%assume_role 
                                      'arn:aws:iam::XXXXXXXXXXXX:role/AWSGlueServiceRole' 
                                      E.g. {credentials}
                                      %%assume_role
                                      {
                                            "aws_access_key_id" : "XXXXXXXXXXXX",
                                            "aws_secret_access_key" : "XXXXXXXXXXXX",
                                            "aws_session_token" : "XXXXXXXXXXXX"
                                       }
----

                                      
## Magic for Spark Sessions (ETL & Streaming)

----
    %worker_type        String        Set the type of instances the session will use as workers. 
    %connections        List          Specify a comma separated list of connections to use in the session.
    %extra_py_files     List          Comma separated list of additional Python files From S3.
    %extra_jars         List          Comma separated list of additional Jars to include in the cluster.
    %spark_conf         String        Specify custom spark configurations for your session. 
                                      E.g. %spark_conf spark.serializer=org.apache.spark.serializer.KryoSerializer
----

## Action Magic

----

    %%sql               String        Run SQL code. All lines after the initial %%sql magic will be passed
                                      as part of the SQL code.  
    %matplot      Matplotlib figure   Visualize your data using the matplotlib library.
                                      E.g. 
                                      import matplotlib.pyplot as plt
                                      # Set X-axis and Y-axis values
                                      x = [5, 2, 8, 4, 9]
                                      y = [10, 4, 8, 5, 2]
                                      # Create a bar chart 
                                      plt.bar(x, y) 
                                      # Show the plot
                                      %matplot plt    
    %plotly            Plotly figure  Visualize your data using the plotly library.
                                      E.g.
                                      import plotly.express as px
                                      #Create a graphical figure
                                      fig = px.line(x=["a","b","c"], y=[1,3,2], title="sample figure")
                                      #Show the figure
                                      %plotly fig

  
                
----



####  Run this cell to set up and start your interactive session.


In [15]:
%idle_timeout 120
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import boto3

from pyspark.sql import functions as F
from pyspark.sql.types import *

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session d9f5a7b6-2585-4aa4-b7f4-93f1e07d9277.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is 120 minutes.
idle_timeout has been set to 120 minutes.


You are already connected to a glueetl session d9f5a7b6-2585-4aa4-b7f4-93f1e07d9277.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session d9f5a7b6-2585-4aa4-b7f4-93f1e07d9277.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session d9f5a7b6-2585-4aa4-b7f4-93f1e07d9277.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 2
Setting new number of workers to: 2



In [3]:
print("Hello world")

Hello world


### Step 1: Create orders,customers,products, order_items dataframes

In [4]:
orders_df = spark.read.option("mergeSchema", True) \
    .json("s3://capstone-project-bucket-12345/raw/orders/")
order_items_df = spark.read.option("mergeSchema", True) \
    .json("s3://capstone-project-bucket-12345/raw/order_items/")
customers_df = spark.read.option("header", True).option("mergeSchema", True) \
    .csv("s3://capstone-project-bucket-12345/raw/customers/")
products_df = spark.read.option("header", True).option("mergeSchema", True) \
    .csv("s3://capstone-project-bucket-12345/raw/products/")

In [5]:
order_items_df.show()

+----------+---------+----------+--------+----------+----------+
|line_total| order_id|product_id|quantity|unit_price|        dt|
+----------+---------+----------+--------+----------+----------+
|    433.25|D1O000001|     P0069|       1|    433.25|2026-07-11|
|    363.69|D1O000001|     P0731|       3|    121.23|2026-07-11|
|      15.3|D1O000001|     P0260|       3|       5.1|2026-07-11|
|    737.24|D1O000002|     P0641|       2|    368.62|2026-07-11|
|    328.48|D1O000002|     P0768|       1|    328.48|2026-07-11|
|    528.38|D1O000002|     P0500|       2|    264.19|2026-07-11|
|    269.18|D1O000003|     P0787|       1|    269.18|2026-07-11|
|   1129.44|D1O000003|     P0424|       4|    282.36|2026-07-11|
|     578.7|D1O000004|     P0431|       2|    289.35|2026-07-11|
|    954.81|D1O000004|     P0042|       3|    318.27|2026-07-11|
|    499.93|D1O000004|     P0482|       1|    499.93|2026-07-11|
|   1572.92|D1O000005|     P0146|       4|    393.23|2026-07-11|
|     206.5|D1O000005|   

### Step 2: Data Handling checks

In [6]:
def handle_nulls(df):
    #print("Entering into Null handling")
    return df.replace("", None)

In [7]:
def deduplicate(df, key_cols):
    #print('Entering into duplication handling')
    return df.dropDuplicates(key_cols)

In [8]:
def check_completeness(df, required_cols):
    valid = df
    for col in required_cols:
        if col in df.columns:
            valid = valid.filter(F.col(col).isNotNull())

    # Quarantine = rows that fail completeness
    quarantine = df
    for col in required_cols:
        if col in df.columns:
            quarantine = quarantine.filter(F.col(col).isNull())

    return valid, quarantine

In [9]:
def publish_quality_score(valid_rows, quarantine_rows, table_name):
    try:
        print(f"\n--- Debugging {table_name} ---")

        # Print schema first
        print("Schema of valid_rows:")
        valid_rows.printSchema()
        print("Schema of quarantine_rows:")
        quarantine_rows.printSchema()

        # Show sample rows
        print("Showing sample of valid_rows:")
        valid_rows.show(5, truncate=False)
        print("Showing sample of quarantine_rows:")
        quarantine_rows.show(5, truncate=False)

        # Count rows
        print("Counting valid_rows...")
        valid_count = valid_rows.count()
        print(f"Valid count = {valid_count}")

        print("Counting quarantine_rows...")
        quarantine_count = quarantine_rows.count()
        print(f"Quarantine count = {quarantine_count}")

        total = valid_count + quarantine_count
        score = (valid_count / total) * 100 if total > 0 else 100

        print(f"[{table_name}] Valid: {valid_count}, Quarantine: {quarantine_count}, Score: {score:.2f}%")

        # Publish to CloudWatch
        print("Publishing metric to CloudWatch...")
        cloudwatch = boto3.client('cloudwatch')
        cloudwatch.put_metric_data(
            Namespace='Capstone/DataQuality',
            MetricData=[{
                'MetricName': f"{table_name}Quality",
                'Value': score,
                'Unit': 'Percent'
            }]
        )
        print("CloudWatch publish succeeded.")

    except Exception as e:
        print(f"❌ Error in publish_quality_score for {table_name}: {e}")
        raise

In [10]:
def process_table(df, key_cols, required_cols, table_name):
    df = handle_nulls(df)
    df = deduplicate(df, key_cols)
    valid, quarantine = check_completeness(df, required_cols)
    publish_quality_score(valid, quarantine, table_name)
    #debug_df(valid, "Orders Valid")
    #debug_df(quarantine, "Orders Quarantine")
    return valid, quarantine

In [11]:
def debug_df(df, label="DataFrame"):
    try:
        print(f"\n--- Debugging {label} ---")

        # If df is None
        if df is None:
            print("❌ DataFrame is None")
            return None, []

        # Print schema
        print("Schema:")
        df.printSchema()

        # Try counting rows
        try:
            count = df.count()
            print(f"Row count = {count}")
        except Exception as e:
            print(f"❌ Error counting rows: {e}")
            count = None

        # Show sample rows only if count > 0
        if count and count > 0:
            print("Showing first 5 rows:")
            df.show(5, truncate=False)

            rows = df.take(5)
            print("Rows as Python objects:")
            for r in rows:
                print(r)
        else:
            print("⚠️ No rows to display")

        return count, []
    except Exception as e:
        print(f"❌ Error while debugging {label}: {e}")
        raise

In [16]:
orders_valid, orders_quarantine = process_table(
    orders_df,
    ["order_id"],
    ["order_id", "customer_id", "order_date"],
    "Orders"
)


--- Debugging Orders ---
Schema of valid_rows:
root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- status: string (nullable = true)
 |-- store_region: string (nullable = true)
 |-- dt: date (nullable = true)

Schema of quarantine_rows:
root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- status: string (nullable = true)
 |-- store_region: string (nullable = true)
 |-- dt: date (nullable = true)

Showing sample of valid_rows:
+------------------------------------+---------+-------------------+---------+------------+----------+
|customer_id                         |order_id |order_ts           |status   |store_region|dt        |
+------------------------------------+---------+-------------------+---------+------------+----------+
|9fb554ec-dd3b-4d98-a812-5931812e8c27|D1O000003|2026-07-11T00:30:00|DELIVERED|East        |2026-07-1

In [17]:
orders_valid.count(), orders_quarantine.count()

(20000, 0)


In [18]:
order_items_valid, order_items_quarantine = process_table(
    order_items_df,
    ["order_id", "product_id"],
    ["order_id", "product_id", "quantity", "unit_price", "line_total"],
    "OrderItems"
)


--- Debugging OrderItems ---
Schema of valid_rows:
root
 |-- line_total: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- dt: date (nullable = true)

Schema of quarantine_rows:
root
 |-- line_total: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- dt: date (nullable = true)

Showing sample of valid_rows:
+----------+---------+----------+--------+----------+----------+
|line_total|order_id |product_id|quantity|unit_price|dt        |
+----------+---------+----------+--------+----------+----------+
|2066.7    |D1O015604|P0273     |5       |413.34    |2026-07-11|
|255.72    |D1O015613|P0612     |3       |85.24     |2026-07-11|
|247.53    |D1O015615|P0702     |1       |247.53    |2026-07-11|
|160.44    |D1O01

In [19]:
order_items_valid.count(), order_items_quarantine.count()

(49966, 0)


In [20]:
customers_valid, customers_quarantine = process_table(
    customers_df,
    ["customer_id"],
    ["customer_id", "name", "email"],
    "Customers"
)


--- Debugging Customers ---
Schema of valid_rows:
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- dt: date (nullable = true)

Schema of quarantine_rows:
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- dt: date (nullable = true)

Showing sample of valid_rows:
+------------------------------------+-------------+-----------------------+-------------------+-------------------+----------+
|customer_id                         |name         |email                  |phone              |city               |dt        |
+------------------------------------+-------------+-----------------------+-------------------+-------------------+----------+
|00103ec0-cc74-4336-8c70-bd1c3367d988|Kaitlyn Blair|ti

In [21]:
customers_valid.count(), customers_quarantine.count()

(4956, 0)


In [22]:
products_valid, products_quarantine = process_table(
    products_df,
    ["product_id"],
    ["product_id", "name", "category"],
    "Products"
)


--- Debugging Products ---
Schema of valid_rows:
root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- dt: date (nullable = true)

Schema of quarantine_rows:
root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- dt: date (nullable = true)

Showing sample of valid_rows:
+----------+--------+----------+-----------+----------+
|product_id|category|unit_price|active_flag|dt        |
+----------+--------+----------+-----------+----------+
|P0001     |Clothing|44.4      |True       |2026-07-11|
|P0002     |Books   |226.91    |False      |2026-07-11|
|P0003     |Books   |222.81    |False      |2026-07-11|
|P0004     |Toys    |173.08    |True       |2026-07-11|
|P0005     |Books   |130.13    |False      |2026-07-11|
+----------+--------+----------+---

In [23]:
products_valid.count(), products_quarantine.count()

(800, 0)


### Step 3: Type Casting

In [24]:
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType
from pyspark.sql import functions as F

# Orders
orders_df = orders_df \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("order_ts", F.col("order_ts").cast(StringType())) \
    .withColumn("status", F.col("status").cast(StringType())) \
    .withColumn("store_region", F.col("store_region").cast(StringType())) \
    .withColumn("dt", F.col("dt").cast(DateType()))

# Order Items
order_items_df = order_items_df \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("product_id", F.col("product_id").cast(StringType())) \
    .withColumn("quantity", F.col("quantity").cast(IntegerType())) \
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType())) \
    .withColumn("line_total", F.col("line_total").cast(DoubleType())) \
    .withColumn("dt", F.col("dt").cast(DateType()))

# Customers
customers_df = customers_df \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("name", F.col("name").cast(StringType())) \
    .withColumn("email", F.col("email").cast(StringType())) \
    .withColumn("dt", F.col("dt").cast(DateType()))

# Products
products_df = products_df \
    .withColumn("product_id", F.col("product_id").cast(StringType())) \
    .withColumn("category", F.col("category").cast(StringType())) \
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType())) \
    .withColumn("active_flag", F.col("active_flag").cast(StringType())) \
    .withColumn("dt", F.col("dt").cast(DateType()))

In [25]:
orders_df.printSchema()
orders_df.show(3)

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- status: string (nullable = true)
 |-- store_region: string (nullable = true)
 |-- dt: date (nullable = true)

+--------------------+---------+-------------------+---------+------------+----------+
|         customer_id| order_id|           order_ts|   status|store_region|        dt|
+--------------------+---------+-------------------+---------+------------+----------+
|7a724c96-9252-48a...|D1O000001|2026-07-11T00:00:00|DELIVERED|       South|2026-07-11|
|c1c00411-a11c-44d...|D1O000002|2026-07-11T00:15:00|DELIVERED|        East|2026-07-11|
|9fb554ec-dd3b-4d9...|D1O000003|2026-07-11T00:30:00|DELIVERED|        East|2026-07-11|
+--------------------+---------+-------------------+---------+------------+----------+
only showing top 3 rows


In [26]:
order_items_df.printSchema()
order_items_df.show(3)

root
 |-- line_total: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- dt: date (nullable = true)

+----------+---------+----------+--------+----------+----------+
|line_total| order_id|product_id|quantity|unit_price|        dt|
+----------+---------+----------+--------+----------+----------+
|    433.25|D1O000001|     P0069|       1|    433.25|2026-07-11|
|    363.69|D1O000001|     P0731|       3|    121.23|2026-07-11|
|      15.3|D1O000001|     P0260|       3|       5.1|2026-07-11|
+----------+---------+----------+--------+----------+----------+
only showing top 3 rows


In [27]:
customers_df.printSchema()
customers_df.show(3)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- dt: date (nullable = true)

+--------------------+-----------------+--------------------+------------------+------------+----------+
|         customer_id|             name|               email|             phone|        city|        dt|
+--------------------+-----------------+--------------------+------------------+------------+----------+
|e954f46b-830c-432...| Mrs. Joan Foster|brownbianca@examp...|   +1-967-682-7141|   Bruceport|2026-07-11|
|7105dda5-d8d2-42c...|Cassandra Aguilar|robertsonsusan@ex...|      876-739-2364|South Nicole|2026-07-11|
|3b128cfa-a99c-4b8...|     Amy Matthews|anthony99@example...|(590)698-9042x6485| North David|2026-07-11|
+--------------------+-----------------+--------------------+------------------+------------+----------+
only showing top 3 rows


In [28]:
products_df.printSchema()
products_df.show(3)

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- dt: date (nullable = true)

+----------+--------+----------+-----------+----------+
|product_id|category|unit_price|active_flag|        dt|
+----------+--------+----------+-----------+----------+
|     P0001|Clothing|      44.4|       True|2026-07-11|
|     P0002|   Books|    226.91|      False|2026-07-11|
|     P0003|   Books|    222.81|      False|2026-07-11|
+----------+--------+----------+-----------+----------+
only showing top 3 rows


### Step 4: Referential Integrity: Orders → Customers


In [29]:
# Referential Integrity: Orders → Customers
valid_customers = customers_valid.select("customer_id").distinct()

orders_valid_ref = orders_valid.join(valid_customers, "customer_id", "inner")
orders_quarantine_ref = orders_valid.join(valid_customers, "customer_id", "left_anti")

# Referential Integrity: Order Items → Products
valid_products = products_valid.select("product_id").distinct()

order_items_valid_ref = order_items_valid.join(valid_products, "product_id", "inner")
order_items_quarantine_ref = order_items_valid.join(valid_products, "product_id", "left_anti")


In [30]:
print("Orders Referential Integrity Check:")
print(f"Valid Orders: {orders_valid_ref.count()}, Quarantined Orders: {orders_quarantine_ref.count()}")
orders_quarantine_ref.show(5, truncate=False)

print("Order Items Referential Integrity Check:")
print(f"Valid Order Items: {order_items_valid_ref.count()}, Quarantined Order Items: {order_items_quarantine_ref.count()}")
order_items_quarantine_ref.show(5, truncate=False)


Orders Referential Integrity Check:
Valid Orders: 0, Quarantined Orders: 20000
+------------------------------------+---------+-------------------+---------+------------+----------+
|customer_id                         |order_id |order_ts           |status   |store_region|dt        |
+------------------------------------+---------+-------------------+---------+------------+----------+
|9fb554ec-dd3b-4d98-a812-5931812e8c27|D1O000003|2026-07-11T00:30:00|DELIVERED|East        |2026-07-11|
|8658f1f2-eae6-49ec-93ec-f45f9733ab69|D1O000008|2026-07-11T01:45:00|SHIPPED  |East        |2026-07-11|
|bf2df4bb-d55b-411e-a2bc-4438c7e07ad0|D1O000013|2026-07-11T03:00:00|PLACED   |North       |2026-07-11|
|39cb70d0-4634-4deb-871a-403568abe05d|D1O000018|2026-07-11T04:15:00|DELIVERED|North       |2026-07-11|
|c4b4cdd7-5315-4e1e-af2e-a21eb60dbd9e|D1O000020|2026-07-11T04:45:00|DELIVERED|North       |2026-07-11|
+------------------------------------+---------+-------------------+---------+------------+------

In [31]:
customers_df.printSchema()
customers_df.show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- dt: date (nullable = true)

+------------------------------------+-----------------+--------------------------+------------------+--------------+----------+
|customer_id                         |name             |email                     |phone             |city          |dt        |
+------------------------------------+-----------------+--------------------------+------------------+--------------+----------+
|e954f46b-830c-432e-93e6-7c71ae0bd2f9|Mrs. Joan Foster |brownbianca@example.org   |+1-967-682-7141   |Bruceport     |2026-07-11|
|7105dda5-d8d2-42cf-9f9e-95d6b930a31b|Cassandra Aguilar|robertsonsusan@example.net|876-739-2364      |South Nicole  |2026-07-11|
|3b128cfa-a99c-4b82-afa3-d4984ff263e6|Amy Matthews     |anthony99@example.org     |(590)698-9042x6485|North David   |2026-07-11|

In [32]:
orders_valid.select("customer_id").distinct().show(5, truncate=False)
customers_valid.select("customer_id").distinct().show(5, truncate=False)

+------------------------------------+
|customer_id                         |
+------------------------------------+
|af1cc4c9-8e80-4e20-b1aa-fec9c689ff66|
|0635bee5-e4b9-4021-90d4-117521747c5c|
|3c5c7c56-2a61-4ec0-a0f3-6648046ca741|
|ebbe2a89-d583-4268-bb7a-5bff211d5af8|
|8d914d47-23b4-4f3d-9edd-d062ed4a2a65|
+------------------------------------+
only showing top 5 rows

+------------------------------------+
|customer_id                         |
+------------------------------------+
|00103ec0-cc74-4336-8c70-bd1c3367d988|
|001b5415-d4ad-4141-9173-7ec6250da0ed|
|00243b5c-9282-42b3-94bf-e9257f3567dd|
|002838ac-4006-4705-a8b8-6776dda9a3f6|
|00337b0e-b437-4e61-96e7-263f1fb8d12c|
+------------------------------------+
only showing top 5 rows


In [33]:
# Just check if any overlap exists
overlap = orders_valid.select("customer_id").intersect(
    customers_valid.select("customer_id")
)

print("Overlap count:", overlap.count())
overlap.show(5, truncate=False)

Overlap count: 0
+-----------+
|customer_id|
+-----------+
+-----------+


In [34]:
from pyspark.sql import functions as F

orders_unique = orders_valid.agg(F.approx_count_distinct("customer_id")).collect()[0][0]
customers_unique = customers_valid.agg(F.approx_count_distinct("customer_id")).collect()[0][0]

print("Approx unique customers in Orders:", orders_unique)
print("Approx unique customers in Customers:", customers_unique)


Approx unique customers in Orders: 21423
Approx unique customers in Customers: 4848


### Step 5: Write Outputs (Partitioned by dt)

In [35]:
# Orders
orders_valid_ref.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/orders/")
orders_quarantine_ref.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/orders/")

# Order Items
order_items_valid_ref.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/order_items/")
order_items_quarantine_ref.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/order_items/")

# Customers
customers_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/customers/")
customers_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/customers/")

# Products
products_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/products/")
products_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/products/")


### Step 6: Commit Job

In [36]:
job.commit()

# Below Code used config dqdl.json file to check the data quality

In [75]:
import json, boto3

s3 = boto3.client("s3")
obj = s3.get_object(Bucket="capstone-project-bucket-12345", Key="config/dqdl_rules.json")
ruleset = json.loads(obj["Body"].read().decode("utf-8"))


In [76]:
ruleset

{'Rules': [{'Table': 'orders', 'Check': 'Completeness', 'Column': 'order_id'}, {'Table': 'orders', 'Check': 'Uniqueness', 'Column': 'order_id'}, {'Table': 'orders', 'Check': 'ReferentialIntegrity', 'Column': 'customer_id', 'ReferencedColumn': 'customers.customer_id'}, {'Table': 'order_items', 'Check': 'Completeness', 'Column': 'product_id'}, {'Table': 'order_items', 'Check': 'ReferentialIntegrity', 'Column': 'product_id', 'ReferencedColumn': 'products.product_id'}, {'Table': 'customers', 'Check': 'Completeness', 'Column': 'customer_id'}, {'Table': 'customers', 'Check': 'Uniqueness', 'Column': 'customer_id'}, {'Table': 'products', 'Check': 'Completeness', 'Column': 'product_id'}, {'Table': 'products', 'Check': 'Uniqueness', 'Column': 'product_id'}]}


In [77]:
def get_rules_for_table(ruleset, table_name):
    return {"Rules": [r for r in ruleset["Rules"] if r["Table"] == table_name]}

In [72]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def completeness_check(df, column):
    valid = df.filter(F.col(column).isNotNull())
    quarantine = df.filter(F.col(column).isNull())
    return valid, quarantine

def uniqueness_check(df, column):
    window = Window.partitionBy(column).orderBy(F.lit(1))
    deduped = df.withColumn("rn", F.row_number().over(window))
    valid = deduped.filter(F.col("rn") == 1).drop("rn")
    quarantine = deduped.filter(F.col("rn") > 1).drop("rn")
    return valid, quarantine

def referential_integrity_check(df, column, ref_df, ref_column):
    valid = df.join(ref_df.select(ref_column).distinct(),
                    df[column] == ref_df[ref_column], "inner")
    quarantine = df.join(ref_df.select(ref_column).distinct(),
                         df[column] == ref_df[ref_column], "left_anti")
    return valid, quarantine

In [73]:
def apply_rules(df, ruleset, reference_dfs=None):
    valid = df
    quarantine = df.sparkSession.createDataFrame([], df.schema)

    for rule in ruleset["Rules"]:
        check = rule["Check"]
        col = rule["Column"]

        if check == "Completeness":
            v, q = completeness_check(valid, col)
        elif check == "Uniqueness":
            v, q = uniqueness_check(valid, col)
        elif check == "ReferentialIntegrity":
            ref_table, ref_col = rule["ReferencedColumn"].split(".")
            ref_df = reference_dfs[ref_table]
            v, q = referential_integrity_check(valid, col, ref_df, ref_col)
        else:
            continue

        valid, quarantine = v, quarantine.unionByName(q)

    return valid, quarantine

In [78]:
orders_rules = get_rules_for_table(ruleset, "orders")
orders_valid, orders_quarantine = apply_rules(
    orders_df, orders_rules, reference_dfs={"customers": customers_df}
)

In [79]:
orders_valid.count(), orders_quarantine.count()

(0, 20000)


In [80]:
order_items_rules = get_rules_for_table(ruleset, "order_items")
order_items_valid, order_items_quarantine = apply_rules(
    order_items_df, order_items_rules, reference_dfs={"products": products_df}
)

In [81]:
order_items_valid.count(), order_items_quarantine.count()

(49961, 66)


In [82]:
customers_rules = get_rules_for_table(ruleset, "customers")
customers_valid, customers_quarantine = apply_rules(customers_df, customers_rules)

In [83]:
customers_valid.count(), customers_quarantine.count()

(5000, 50)


In [84]:
products_rules = get_rules_for_table(ruleset, "products")
products_valid, products_quarantine = apply_rules(products_df, products_rules)

In [85]:
products_valid.count(), products_quarantine.count()

(800, 0)


### Referential Integrity check

In [88]:
def referential_integrity_check(df, column, ref_df, ref_column):
    # Keep only the original df columns after join
    valid = df.join(ref_df.select(ref_column).distinct(),
                    df[column] == ref_df[ref_column], "inner") \
              .select(df["*"])

    quarantine = df.join(ref_df.select(ref_column).distinct(),
                         df[column] == ref_df[ref_column], "left_anti") \
                   .select(df["*"])

    return valid, quarantine

In [89]:
orders_valid, orders_quarantine = referential_integrity_check(
    orders_df, "customer_id", customers_df, "customer_id"
)

In [90]:
order_items_valid, order_items_quarantine = referential_integrity_check(
    order_items_df, "product_id", products_df, "product_id"
)

In [94]:
from pyspark.sql import functions as F

# Ensure dt column exists and is string
orders_valid = orders_valid.withColumn("dt", F.col("dt").cast("string"))
orders_quarantine = orders_quarantine.withColumn("dt", F.col("dt").cast("string"))

### Writing to the output curated/quarantine bucket

In [96]:
orders_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/orders/")

orders_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/orders/")

In [97]:
orders_valid.count(), orders_quarantine.count()

(0, 20000)


In [98]:
order_items_valid.count(), order_items_quarantine.count()

(49961, 66)


In [99]:
# Order Items
order_items_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/order_items/")
order_items_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/order_items/")

In [100]:
products_valid.count(), products_quarantine.count()

(800, 0)


In [101]:
# Products
products_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/products/")
products_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/products/")

In [102]:
# Customers
customers_valid.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/curated/customers/")
customers_quarantine.write.mode("overwrite").partitionBy("dt") \
    .parquet("s3://capstone-project-bucket-12345/quarantine/customers/")

In [103]:
job.commit()